In [ ]:
import os
import kaggle_evaluation.aimo_3_inference_server
import pandas as pd
import polars as pl
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# Import the inference server specifically for the AIMO 3 competition
import kaggle_evaluation.aimo_3_inference_server

# Path to your model files (weights, config, tokenizer)
# You need to change this to the actual path in your Kaggle environment
MODEL_PATH = "/kaggle/input/YOUR_MODEL_DATASET/YOUR_MODEL_FOLDER"

# Define the maximum number of new tokens the model should generate
MAX_NEW_TOKENS = 512

# Regex pattern to find integers (including negative ones) in the text
_INT_RE = re.compile(r"-?\d+")

def extract_final_int(text: str) -> int:
    """
    Parses the model's output text to find the last integer.
    Returns that integer modulo 100,000 (valid answer range).
    """
    nums = _INT_RE.findall(text)
    if not nums:
        return 0
    # The competition typically requires the answer modulo 100000
    return int(nums[-1]) % 100000

def make_prompt(problem: str) -> str:
    """
    Creates the prompt to feed into the language model.
    Structure: Instructions + Problem + 'Final answer:'
    """
    # Keep fixed for baseline comparisons
    return (
        "Solve the following math problem.\n"
        "Return ONLY the final answer as an integer from 0 to 99999.\n\n"
        f"Problem:\n{problem}\n\n"
        "Final answer:"
    )


class Model:
    """
    A class to manage loading the LLM and running inference.
    """
    def __init__(self):
        self.tokenizer = None
        self.model = None

    def load(self):
        """
        Loads the pre-trained model and tokenizer from MODEL_PATH.
        """
        print("Loading model...")
        # Load tokenizer. trust_remote_code=True allows custom code in the model repo.
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True, trust_remote_code=True)
        # Ensure the tokenizer has a pad token
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load the causal language model
        # Use float16 precision if a GPU is available to save memory
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
        )
        self.model.eval() # Set to evaluation mode (disable dropout, etc.)

    @torch.inference_mode() # Disable gradient calculation for faster inference
    def predict(self, problem: str) -> int:
        """
        Takes a math problem string, generates a response, and extracts the answer.
        """
        if self.model is None:
            self.load()

        prompt = make_prompt(problem)
        # Tokenize prompt and move to the same device as the model (e.g., GPU)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # Generate tokens
        out = self.model.generate(
            **inputs,
            do_sample=False,        # Greedy decoding (always pick most likely token)
            temperature=0.0,        # Zero temperature (deterministic)
            top_p=1.0,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Decode tokens to string
        text = self.tokenizer.decode(out[0], skip_special_tokens=True)
        # Extract the integer answer
        return extract_final_int(text)


# Create a global instance of the Model class
model = Model()


def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame | pd.DataFrame:
    """
    The main callback function for the inference server.
    Arguments:
        id_: Polars Series containing the problem ID.
        problem: Polars Series containing the problem text.
    Returns:
        Polars DataFrame with 'id' and 'answer'.
    """
    # Unpack values (gateway sends 1-row series for each request)
    id_val = id_.item(0)
    problem_text = problem.item(0)

    # Use the model to predict the answer
    ans = model.predict(problem_text)

    # AIMO3 format requires returning a DataFrame with columns 'id' and 'answer'
    return pl.DataFrame({"id": [id_val], "answer": [ans]})


# Initialize the inference server with our predict function
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

# Check environment variable to decide between submission mode and local test mode
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # This runs when you submit to the competition
    inference_server.serve()
else:
    # This runs in your interactive session to test with a dummy file
    inference_server.run_local_gateway(
        ("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",)
    )